# deepEmulator — Pokemon Coral on Colab

First Colab training run for Pokemon Coral (Crystal-engine romhack). Designed to be smooth, resumable across session disconnects, and honest about what to expect.

## Two success regimes

**With `states/coral_init.state`** (recommended): agent starts in the overworld with a starter → tutorial-phase rewards fire → demonstrable learning in <2 hours wall-time. `MeanReward` grows over episodes, exploration count climbs.

**Without init.state** (fallback): agent boots cold to title screen → boot-phase reward gives +0.01 per action → DDQN trains but converges to a degenerate constant Q. Pipeline is smooth, no policy improvement. Useful for verifying the plumbing only.

**The init.state is the load-bearing factor.** ~5 minutes of you playing through Coral's intro once is the difference between "watched a flat line" and "watched an agent learn."

In [ ]:
# Cell 2 — GPU + dependency install. Surfaces problems before anything else.
!nvidia-smi -L || echo 'NO GPU — runtime > change runtime type > T4 GPU'
!pip install -q pyboy torch numpy

In [ ]:
# Cell 3 — Drive sync + repo install.
# This avoids needing the repo to exist on GitHub. Two paths supported below:
from google.colab import drive
drive.mount('/content/drive')

# Path A (recommended): sync the local repo folder into Drive once, then:
!ls /content/drive/MyDrive/deepEmulator | head -10
!pip install -q /content/drive/MyDrive/deepEmulator

# Path B (alternative): if you pushed to GitHub, uncomment:
# !pip install -q git+https://github.com/<your-fork>/deepEmulator.git

In [ ]:
# Cell 4 — Pre-flight dry-run. Verifies every layer in ~30 seconds before you commit
# to a 30-min training cell. Hard-exits with a clear message on any failure.
import importlib.util
import torch
from pathlib import Path

assert importlib.util.find_spec('pyboy') is not None, 'pyboy not installed (Cell 2 failed?)'
assert torch.cuda.is_available(), 'no CUDA — runtime > change runtime type > T4 GPU'
print(f'[dry-run] cuda OK — {torch.cuda.get_device_name(0)}')

from deepEmulator.cartridges import pokemon_coral  # registers POKEMON CORAL
from deepEmulator.cartridges.pokemon_crystal import dump_state
from deepEmulator.core import registry
from deepEmulator.platforms.gameboy import PyBoyEnv
from deepEmulator.agents.ddqn_torch import DDQNAgent, DDQNConfig

ROM = Path('/content/drive/MyDrive/deepEmulator/roms/PokemonCoral.gbc')
INIT_STATE = Path('/content/drive/MyDrive/deepEmulator/states/coral_init.state')
assert ROM.exists(), f'ROM not in Drive: {ROM}'
print(f'[dry-run] ROM ok ({ROM.stat().st_size:,} bytes)')
print(f'[dry-run] init.state present: {INIT_STATE.exists()} ({INIT_STATE})')

adapter = registry.get('POKEMON CORAL')(init_state=INIT_STATE if INIT_STATE.exists() else None)
env = PyBoyEnv(adapter, rom_path=ROM,
               init_state=INIT_STATE if INIT_STATE.exists() else None,
               headless=True, max_steps=20)
obs, info = env.reset()
assert obs.shape == (3, 72, 80), f'bad obs shape {obs.shape}'
print(f'[dry-run] env.reset OK; obs.shape={obs.shape}')

# Print RAM state so you can sanity-check the # VERIFY addresses.
# Nonsense values (badges=255, party_count=42) mean an address constant is wrong.
ram = dump_state(env.pyboy)
print(f'[dry-run] RAM at reset: {ram}')

for i in range(5):
    obs2, r, term, trunc, info = env.step(i % env.action_space.n)
    assert isinstance(r, float) and r == r, f'reward NaN at step {i}: {r}'
print(f'[dry-run] 5 env.step calls OK')

agent = DDQNAgent(obs.shape, env.action_space.n,
                  config=DDQNConfig(burnin=2, batch_size=2, deque_size=10, learn_every=1, sync_every=100))
for _ in range(5):
    a = agent.act(obs); obs2, r, _, _, _ = env.step(a)
    agent.cache(obs, obs2, a, r, False); obs = obs2
q, loss = agent.learn()
assert q is not None and loss is not None, 'DDQN gradient step did not fire'
assert agent.memory[0][0].dtype == torch.uint8, 'replay buffer not uint8 (F9.7 not applied?)'
print(f'[dry-run] DDQN gradient ok (q={q:.3f}, loss={loss:.4f}); buffer dtype OK')
env.close()
print('[dry-run] ✅ all checks passed — safe to run Cell 5')

In [ ]:
# Cell 5 — Actual training. ~30-60 min on free T4 for 100K steps.
# Re-running this cell after a session disconnect auto-resumes from the latest bundle
# in MyDrive/deepEmulator/checkpoints/pokemon_coral/.
from pathlib import Path
from deepEmulator.training.colab_train import run_in_colab

# F9 safety: pass init_state only if the file actually exists in Drive.
# Without it, agent boots cold and stays in boot phase (plumbing-only run).
_state = Path('/content/drive/MyDrive/deepEmulator/states/coral_init.state')
init_state = 'deepEmulator/states/coral_init.state' if _state.exists() else None
print(f'[train] init.state: {"present" if init_state else "MISSING — will train in boot-only regime"}')

run_in_colab(
    cartridge='POKEMON CORAL',
    rom='deepEmulator/roms/PokemonCoral.gbc',
    init_state=init_state,
    steps=100_000,
    max_episode_steps=2048,
    save_every=10_000,
    resume=True,
)

In [ ]:
# Cell 6 — Metrics inspection.
import pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

root = Path('/content/drive/MyDrive/deepEmulator/checkpoints/pokemon_coral')
latest = Path((root / 'latest.txt').read_text().strip())
df = pd.read_csv(latest / 'metrics.tsv', sep='\t')
print(f'run: {latest}\nepisodes logged: {len(df)}')
df.tail(20)

In [ ]:
# Plot the curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df.plot(x='Step', y='MeanReward', ax=axes[0], title='Mean Reward per Episode')
df.plot(x='Step', y='MeanLoss', ax=axes[1], title='Mean Loss (0 until burnin)')
df.plot(x='Step', y='MeanQValue', ax=axes[2], title='Mean Q Value')
plt.tight_layout(); plt.show()

## Honest interpretation guide

### What success looks like

**Plumbing success** (any run, with or without init.state):
- `MeanLoss > 0` after step 1000 → DDQN has finished burnin and is actually training.
- Bundle saved to Drive every 10K steps.
- `latest.txt` always points at the latest run.
- Re-running Cell 5 after a disconnect resumes cleanly.

**Learning success** (only achievable with `coral_init.state`):
- `MeanReward` grows over episodes (above the boot floor of ~20 per episode).
- `MeanQValue` doesn't collapse to ~0.1 — it grows as the agent discovers higher-value states.
- Trajectory CSVs (`<run>/trajectories/episode_*.csv.gz`) show varying `(x, y, map_id)` — not stuck at `(0, 0, 0)`.

### What failure looks like

**`MeanLoss == 0` after step 1000**: burnin not yet complete, or no `learn()` calls happening. Check that `DDQNConfig.burnin` is reasonable.

**`MeanLoss == NaN`**: catastrophic. Causes:
- Learning rate too high → reduce `DDQNConfig.learning_rate`.
- Reward exploding → check `dump_state` output for bogus values (wrong `# VERIFY` RAM addresses).
- Replay buffer dtype corruption → ensure F9.7 fix is in place (Cell 4 dry-run asserts this).

**`MeanQValue` grows unboundedly**: γ too close to 1, or LR too high.

**`MeanReward` flat at ~20 forever** (no init.state): expected. Agent is in degenerate boot regime. Either record an init.state (see notebook 07) or accept this as a plumbing-only validation run.

**Cell 4 dry-run prints nonsense RAM** (e.g. `johto_badge_count: 8` at the title screen): one or more `# VERIFY` addresses in `cartridges/pokemon_crystal.py` are wrong for Coral. Edit those constants and re-run dry-run.